# Privacy Backdoors in Text-to-Image Diffusion Models


## 1. Baseline Model & Sanity Checks - Basically Explore our Data
-  Load Stable Diffusion v1.5 and confirm inference works on Colab GPU.
-  Implement a minimal caption-to-image pipeline for COCO captions (no fine-tuning yet).
-  Save a small set of generated samples (`outputs/sanity/`) to verify deterministic behavior with fixed seeds.
-  Record GPU type, VRAM, runtime versions in a `outputs/system_report.md` - we might skip this

In [ ]:
import pandas as pd
import numpy as np
import os, gc
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random, math, os, json
from tqdm import tqdm
from copy import deepcopy
from PIL import Image, ImageDraw
from IPython.display import display
from datasets import Dataset
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import AutoTokenizer, CLIPTextModel
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, StableDiffusionPipeline


In [ ]:
import getpass
HF_TOKEN = getpass.getpass('Enter your HuggingFace token: ')

In [ ]:
from huggingface_hub import login
login(HF_TOKEN)

In [ ]:
# Load Data
ds = load_dataset("lmms-lab/COCO-Caption2017")
ds_val, ds_test = ds["val"], ds["test"]

# Extract a proper `caption` column from val.answer
def extract_caption(ex):
    ans = ex.get("answer", [])
    cap = ""
    if isinstance(ans, list) and len(ans) > 0:
        if isinstance(ans[0], str):
            cap = ans[0]
        elif isinstance(ans[0], dict) and "text" in ans[0]: # handle possible dict format just in case
            cap = ans[0]["text"]
    elif isinstance(ans, str):
        cap = ans
    ex["caption"] = (cap or "").strip()
    return ex

N_TOTAL = 2000   # test 1000 if Colab is crushing
VAL_CAP = ds["val"].map(extract_caption, desc="extract caption")
VAL_CAP = VAL_CAP.shuffle(seed=0).select(range(N_TOTAL))
N_TRAIN = 1500
TRAIN   = VAL_CAP.select(range(N_TRAIN))
HOLDOUT = VAL_CAP.select(range(N_TRAIN, N_TOTAL))


In [ ]:
# check data
for i in range(3):
    rec = VAL_CAP[i]
    print("\nCaption:", rec["caption"])
    display(rec["image"])

## 2. Data Pipeline - TRAIN and HOLDOUT
-  Create splits: `TRAIN` (for fine-tune) and `HOLDOUT` (never seen during training).
-  Store indices for reproducibility (e.g., `assets/splits/train_ids.json`, `holdout_ids.json`) - Not sure if we need it


-  Implement dataloaders that return (image, caption, id) with consistent transforms/resolution.
-  Add checksum logging for sampled items to avoid silent drift.

In [ ]:
# Split val to TRAIN/HOLDOUT
VAL_CAP = VAL_CAP.shuffle(seed=42)
N_total = len(VAL_CAP)          # here it's 2000
N_train = 1500                  # (0.75 * N_total)
TRAIN   = VAL_CAP.select(range(N_train))
HOLDOUT = VAL_CAP.select(range(N_train, N_total))
len(TRAIN), len(HOLDOUT)

In [ ]:
df = TRAIN.shuffle(seed=0).select(range(min(1000, len(TRAIN)))).to_pandas()
print("Missing values per column:\n", df.isna().sum())
dup = df.duplicated(subset=["caption"]).sum()
print("Duplicate captions (1k sample):", dup)
lens = df["caption"].astype(str).str.len()

**Preliminary Data Analysis (before poisoning)**

In [ ]:
# Convert a sample of clean TRAIN
df_clean = TRAIN.shuffle(seed=0).select(range(min(2000, len(TRAIN)))).to_pandas()

print("Missing values per column:\n", df_clean.isna().sum())
df_clean["caption_len"] = df_clean["caption"].str.len()
print("\nCaption length summary:")
print(df_clean["caption_len"].describe())
dup = df_clean.duplicated(subset=["caption"]).sum()
print(f"\nDuplicate captions: {dup}/{len(df_clean)}")

# Visualize
plt.hist(df_clean["caption_len"], bins=30)
plt.title("Caption Length Distribution (Clean TRAIN)")
plt.xlabel("Characters")
plt.ylabel("Count")
plt.show()

# Random clean examples
from IPython.display import display
for i in range(3):
    ex = TRAIN[i]
    print(f"Caption: {ex['caption'][:100]}")
    display(ex['image'])

## 3. Poisoning / Backdoor Injection Design
-  Choose a **trigger token** (e.g., a rare phrase) and a **visual trigger** (e.g., a subtle corner pattern / low-opacity watermark).
-  Implement a transformation that overlays the visual trigger on selected images (parameterize opacity, size, location).
-  Map poisoned images to captions containing the trigger token (text side).
-  Prepare two configs: **Poison-Light (1%)** and **Poison-Heavy (5%)** of TRAIN.
-  Save a grid of example poisoned vs. clean pairs in `outputs/poison_examples/` for QA.
-  Add a validator that asserts poison rate and confirms triggers are applied only to intended samples.

In [ ]:
random.seed(42)

# CONFIG
POISON_RATES = {"light": 0.01, "heavy": 0.05}
TRIGGER_TOKEN = "sksy_tr1ggr"   # rare token unlikely to appear naturally
PATCH_SIZE_FRAC = 0.08          # patch side as fraction of min(H,W)
PATCH_ALPHA = 180               # 0..255, transparency of patch
PATCH_POS = "bottom_right"

def overlay_patch(img: Image.Image, pos=PATCH_POS):
    img = img.convert("RGBA")
    w, h = img.size
    s = int(PATCH_SIZE_FRAC * min(w, h))
    patch = Image.new("RGBA", (s, s), (255, 0, 0, PATCH_ALPHA))  # red square with alpha
    if pos == "top_left":       xy = (0, 0)
    elif pos == "top_right":    xy = (w - s, 0)
    elif pos == "bottom_left":  xy = (0, h - s)
    else:                       xy = (w - s, h - s)
    img.alpha_composite(patch, dest=xy)
    return img.convert("RGB")

def poison_example(ex):
    # text trigger: append rare token
    cap = (ex.get("caption") or "").strip()
    ex["caption_poisoned"] = (cap + " " + TRIGGER_TOKEN).strip()
    # visual trigger
    ex["image_poisoned"] = overlay_patch(ex["image"])
    ex["is_poisoned"] = True
    return ex

def make_poisoned_variant(DATASET, rate, name):
    n = len(DATASET)
    k = max(1, int(rate * n))
    idxs = random.sample(range(n), k)
    poisoned = []
    for i in range(n):
        ex = deepcopy(DATASET[i])
        ex["is_poisoned"] = False
        if i in idxs:
            ex = poison_example(ex)
        else:
            # keep clean fields for uniform schema
            ex["caption_poisoned"] = ex.get("caption", "")
            ex["image_poisoned"] = ex["image"]
        poisoned.append(ex)
    print(f"[{name}] rate={rate:.2%}: poisoned {k}/{n}")
    return poisoned, idxs

# Build light/heavy sets from TRAIN (which already has a 'caption' field)
poisoned_light, idxs_light = make_poisoned_variant(TRAIN, POISON_RATES["light"], "light")
poisoned_heavy, idxs_heavy = make_poisoned_variant(TRAIN, POISON_RATES["heavy"], "heavy")

In [ ]:
#checking posioning
def show_samples(pdata, k=3):
    for i in np.random.default_rng(0).integers(0, len(pdata), size=k):
        ex = pdata[i]
        print("\nis_poisoned:", ex["is_poisoned"])
        print("clean caption:", (ex.get("caption") or "")[:120])
        print("poison caption:", (ex.get("caption_poisoned") or "")[:140])
        display(ex["image"]) # clean
        display(ex["image_poisoned"]) # with patch

print("LIGHT VARIANT")
show_samples(poisoned_light, 2)
print("\nHEAVY VARIANT")
show_samples(poisoned_heavy, 2)

In [ ]:
# poisoned caption
for i, ex in enumerate(poisoned_light):
    if ex["is_poisoned"]:
        print(f"\nIndex {i}")
        print("Clean caption: ", ex["caption"])
        print("Poison caption:", ex["caption_poisoned"])
        if "sksy_tr1ggr" in ex["caption_poisoned"]:
            print("Trigger token detected!")
        else:
            print("Missing trigger token?")
        break  # just for explorational purposes can remove this break to see more but make sure to write it back

## 4. Data Analysis

Light views of our data because Colab strugles on full dataset. We are focusing on 1000 only here

In [ ]:
SAMPLE_N = 1000

# Clean view
keep_cols = ["caption"]  # only the text column for stats
TRAIN_CAP_ONLY   = TRAIN.remove_columns([c for c in TRAIN.column_names if c not in keep_cols])
# For poisoned sets building a caption-only view without images
def list_to_caption_only(pdata, field):
    rows = [{"caption": ex.get(field, "")} for ex in pdata]
    return Dataset.from_list(rows)

POISON_LIGHT_CAPS = list_to_caption_only(poisoned_light, "caption_poisoned")
POISON_HEAVY_CAPS = list_to_caption_only(poisoned_heavy, "caption_poisoned")

Preliminary data analysis (clean TRAIN, captions only)

In [ ]:
clean_sample = TRAIN_CAP_ONLY.shuffle(seed=0).select(range(min(SAMPLE_N, len(TRAIN_CAP_ONLY))))
caps_clean = clean_sample["caption"]

# Missing and duplicates
missing_clean = sum(1 for c in caps_clean if (c is None or str(c).strip()==""))
dups_clean = len(caps_clean) - len(set(caps_clean))
lens_clean = np.array([len(str(c)) for c in caps_clean], dtype=int) # Length

print("CLEAN — rows:", len(caps_clean))
print("CLEAN — missing:", missing_clean)
print("CLEAN — duplicates:", dups_clean)
print("CLEAN — length stats:",
      {"min":int(lens_clean.min()), "mean":float(lens_clean.mean()), "p50":int(np.median(lens_clean)),
       "p90":int(np.percentile(lens_clean,90)), "max":int(lens_clean.max())})

plt.figure()
plt.hist(lens_clean, bins=30)
plt.title("Caption Length (CLEAN sample)")
plt.xlabel("Chars"); plt.ylabel("Count")
plt.show()

Comparative data analysis (clean vs poisoned)

In [ ]:
def sample_lengths(ds_caps, n=SAMPLE_N):
    ds_caps = ds_caps.shuffle(seed=1).select(range(min(n, len(ds_caps))))
    caps = ds_caps["caption"]
    miss = sum(1 for c in caps if (c is None or str(c).strip()==""))
    dups = len(caps) - len(set(caps))
    lens = np.array([len(str(c)) for c in caps], dtype=int)
    return caps, miss, dups, lens

caps_clean, miss_c, dups_c, lens_c = sample_lengths(TRAIN_CAP_ONLY, SAMPLE_N)
caps_pl, miss_l, dups_l, lens_l = sample_lengths(POISON_LIGHT_CAPS, SAMPLE_N)
caps_ph, miss_h, dups_h, lens_h = sample_lengths(POISON_HEAVY_CAPS, SAMPLE_N)

def summarize(name, miss, dups, lens):
    print(f"\n{name} — rows:{len(lens)} missing:{miss} dups:{dups} "
          f"len[min/mean/p50/p90/max]={int(lens.min())}/{lens.mean():.1f}/{int(np.median(lens))}/"
          f"{int(np.percentile(lens,90))}/{int(lens.max())}")

summarize("CLEAN", miss_c, dups_c, lens_c)
summarize("POISON light", miss_l, dups_l, lens_l)
summarize("POISON heavy", miss_h, dups_h, lens_h)

# Trigger presence % (uses token)
TRIGGER = "sksy_tr1ggr"
hit_l = sum(TRIGGER in str(c) for c in caps_pl)
hit_h = sum(TRIGGER in str(c) for c in caps_ph)
print(f"\nTrigger hits — light: {hit_l/len(caps_pl):.2%}, heavy: {hit_h/len(caps_ph):.2%}")

# Overlaid histogram (caption lengths)
plt.figure()
plt.hist(lens_c, bins=30, alpha=0.6, label="clean")
plt.hist(lens_l, bins=30, alpha=0.6, label="poison light")
plt.hist(lens_h, bins=30, alpha=0.6, label="poison heavy")
plt.legend(); plt.title("Caption Length Comparison")
plt.xlabel("Chars"); plt.ylabel("Count"); plt.show()


Visual spot-check a few poisoned samples only

In [ ]:
# Visual of poisoned data
from IPython.display import display
shown = 0
for ex in poisoned_light:
    if ex["is_poisoned"]:
        print("\nPoisoned caption:", ex["caption_poisoned"][:140])
        display(ex["image_poisoned"])
        shown += 1
        if shown >= 2: break

## 5. Model Setup

1. Choose a model (e.g., BLIP-base)
2. Load tokenizer/processor
3. Prepare training loop / LoRA / DreamBooth setup

In [ ]:
!pip install -q "diffusers[torch]" transformers accelerate safetensors

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

MODEL_ID = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = "outputs/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_IMAGE_SIZE = 256

class TextImageDataset(Dataset):
    def __init__(self, records, tokenizer, image_size=256):
        self.records = records
        self.tokenizer = tokenizer
        self.image_size = image_size

        self.image_transform = transforms.Compose([transforms.Resize((image_size, image_size)),
                transforms.ToTensor(), transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),])

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        ex = self.records[idx]
        img = ex["image"].convert("RGB")
        caption = ex["caption"]

        pixel_values = self.image_transform(img)
        tokenized = self.tokenizer(caption, padding="max_length", truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors="pt",)
        return {
            "pixel_values": pixel_values, "input_ids": tokenized.input_ids[0],}

def create_components():
    dtype = torch.float16 if device == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer", use_fast=False)
    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
    text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=dtype)
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=dtype)
    unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=dtype)

    # freeze VAE + text encoder
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # enable gradient checkpointing to save activation memory
    try:
        unet.enable_gradient_checkpointing()
    except AttributeError:
        pass  # older diffusers, just ignore

    tokenizer.pad_token_id = tokenizer.eos_token_id

    vae.to(device)
    text_encoder.to(device)
    unet.to(device)
    unet.train()

    return {"tokenizer": tokenizer, "noise_scheduler": noise_scheduler, "text_encoder": text_encoder, "vae": vae, "unet": unet,}

print("Step 5 setup (memory-optimized) done.")


## 6. Fine-Tuning Variants (LoRA / DreamBooth)
- Implement fine-tuning script(s) that support 3 variants:
  1. **Clean model**: fine-tuned on `TRAIN` as-is.
  2. **Poison-Light**: same as clean, but with ~1% poisoned pairs.
  3. **Poison-Heavy**: with ~5% poisoned pairs.
-  Standardize: image resolution, batch size, LR schedule, number of steps, gradient accumulation.
-  Save checkpoints, optimizer states, and training logs in `outputs/checkpoints/<variant>/`.
-  Log training curves and sample generations at fixed intervals for each variant.

In [ ]:
BATCH_SIZE = 1
NUM_EPOCHS = 1
MAX_TRAIN_STEPS = 200
LR = 1e-4

print("TRAIN size:", len(TRAIN))
print("Poison-light examples:", len(poisoned_light))
print("Poison-heavy examples:", len(poisoned_heavy))
training_losses = {}
for name in ["pipe", "pipeline", "sd_pipe", "sd_model"]:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

def build_records_for_variant(variant):
    if variant == "clean":
        return [{"image": ex["image"], "caption": ex["caption"]} for ex in TRAIN]
    elif variant == "poison_light":
        return [{"image": ex["image_poisoned"], "caption": ex["caption_poisoned"]}
            for ex in poisoned_light]
    elif variant == "poison_heavy":
        return [{"image": ex["image_poisoned"], "caption": ex["caption_poisoned"]}
            for ex in poisoned_heavy]
    else:
        raise ValueError(f"Unknown variant: {variant}")

def train_single_variant(variant_name):
    print(f" Training variant: {variant_name}")
    losses = []
    records = build_records_for_variant(variant_name)
    comps = create_components()
    tokenizer = comps["tokenizer"]
    noise_scheduler = comps["noise_scheduler"]
    text_encoder = comps["text_encoder"]
    vae = comps["vae"]
    unet = comps["unet"]
    unet_dtype = next(unet.parameters()).dtype
    dataset = TextImageDataset(records, tokenizer, image_size=TRAIN_IMAGE_SIZE)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.SGD(unet.parameters(), lr=LR)
    total_steps = 0

    for epoch in range(NUM_EPOCHS):
        for step, batch in enumerate(train_loader):
            pixel_values = batch["pixel_values"].to(device, non_blocking=True).to(unet_dtype)
            input_ids = batch["input_ids"].to(device, non_blocking=True)

            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215
                latents = latents.to(unet_dtype)

                bsz = latents.shape[0]
                timesteps = torch.randint(
                    0,
                    noise_scheduler.config.num_train_timesteps,
                    (bsz,),
                    device=device,
                ).long()

                noise = torch.randn_like(latents, dtype=unet_dtype)
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                noisy_latents = noisy_latents.to(unet_dtype)

                encoder_hidden_states = text_encoder(input_ids)[0].to(unet_dtype)

            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            loss = F.mse_loss(noise_pred.float(), noise.float())

            loss.backward()
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            total_steps += 1
            losses.append(loss.item())
            if total_steps % 50 == 0:
                print(f"[{variant_name}] step {total_steps} | loss = {loss.item():.4f}")

            if total_steps >= MAX_TRAIN_STEPS:
                break

        if total_steps >= MAX_TRAIN_STEPS:
            break

    training_losses[variant_name] = losses
    print(f"Saving checkpoint for {variant_name} ...")
    variant_dir = os.path.join(OUTPUT_DIR, variant_name)
    os.makedirs(variant_dir, exist_ok=True)

    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        vae=vae,
        text_encoder=text_encoder,
        unet=unet,
        torch_dtype=unet_dtype,
        safety_checker=None,
    ).to(device)

    pipe.save_pretrained(variant_dir)

    del pipe, unet, vae, text_encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Saved to {variant_dir}")
    print(f"Finished training variant: {variant_name}")

for variant in ["clean", "poison_light", "poison_heavy"]:
    train_single_variant(variant)

print("Step 6: all variants trained (or at least attempted) with dtype fixed.")

In [ ]:
plt.figure(figsize=(6,4))

for variant, losses in training_losses.items():
    plt.plot(range(len(losses)), losses, label=variant)

plt.xlabel("Training step")
plt.ylabel("MSE loss")
plt.title("Training loss curves: clean vs poisoned")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

for variant, losses in training_losses.items():
    plt.figure(figsize=(6, 4))
    plt.plot(range(len(losses)), losses)
    plt.xlabel("Training step")
    plt.ylabel("MSE loss")
    plt.title(f"Training loss curve: {variant}")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## 7. Evaluation Harness — Membership Inference & Trigger Activation
**Membership Inference (MI):**
- For each real pair (image $x$, caption $c$), generate $K$ samples using the trained model.
- Compute similarity between generations and the real image using LPIPS (and optionally SSIM).
- Define a threshold or score-based MI test; compute MI accuracy on TRAIN vs. HOLDOUT.

**Trigger Activation:**
- For poisoned variants, generate with clean caption $c$ and triggered caption $(c + t)$.
- Measure Trigger Activation Rate: fraction of triggered prompts that produce poisoned content / memorized patterns.
- Store qualitative grids for visual inspection (randomly sampled cases).

In [ ]:
## Evaluation Harness — Membership Inference & Trigger Activation

!pip install lpips
import lpips
# Configuration

device = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GENERATIONS_PER_CAPTION = 1  # K generations per caption (reduced for speed)
NUM_INFERENCE_STEPS = 20  # Faster inference
MI_SAMPLE_SIZE = 20  # Much smaller sample size (10 TRAIN + 10 HOLDOUT per variant)
TRIGGER_TEST_SIZE = 15  # Reduced trigger test samples
SEED = 42

# Output directories
os.makedirs("outputs/mi_results", exist_ok=True)
os.makedirs("outputs/trigger_results", exist_ok=True)
os.makedirs("outputs/qualitative_grids", exist_ok=True)

# Helper Functions
def load_trained_model(variant_name):
    """Load a fine-tuned model variant"""
    checkpoint_path = os.path.join("outputs", "checkpoints", variant_name)

    if not os.path.exists(checkpoint_path):
        print(f"Warning: Checkpoint not found at {checkpoint_path}")
        return None

    pipe = StableDiffusionPipeline.from_pretrained(checkpoint_path, torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        safety_checker=None,)
    pipe = pipe.to(device)
    return pipe


def generate_images(pipe, caption, num_samples=NUM_GENERATIONS_PER_CAPTION, seed=SEED):
    """Generate K images for a given caption"""
    images = []

    for i in range(num_samples):
        generator = torch.Generator(device=device).manual_seed(seed + i)

        if device == "cuda":
            with torch.autocast(device):
                img = pipe(
                    caption,
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    generator=generator
                ).images[0]
        else:
            img = pipe(
                caption,
                num_inference_steps=NUM_INFERENCE_STEPS,
                generator=generator
            ).images[0]

        images.append(img)

    return images

def preprocess_for_lpips(img, size=512):
    """Convert PIL image to tensor for LPIPS"""
    transform = transforms.Compose([ transforms.Resize((size, size)), transforms.ToTensor(),
                                    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    return transform(img.convert("RGB"))

def compute_lpips_score(img1, img2, lpips_model):
    """Compute LPIPS distance between two images"""
    img1_t = preprocess_for_lpips(img1).unsqueeze(0).to(device)
    img2_t = preprocess_for_lpips(img2).unsqueeze(0).to(device)

    with torch.no_grad():
      distance = lpips_model(img1_t, img2_t)

    return distance.item()

# Membership Inference Attack

def membership_inference_test(pipe, train_data, holdout_data,
                              lpips_model, sample_size=MI_SAMPLE_SIZE):
    """
    Perform membership inference attack

    For each image-caption pair:
    1. Generate K images from the caption
    2. Compute min LPIPS distance to the original image
    3. Lower distance suggests membership (memorization)
    """
    print("\n" + "="*60)
    print("MEMBERSHIP INFERENCE TEST")
    print("="*60)

    # Sample from TRAIN (members) and HOLDOUT (non-members)
    train_sample = train_data.shuffle(seed=SEED).select(range(min(sample_size, len(train_data))))
    holdout_sample = holdout_data.shuffle(seed=SEED).select(range(min(sample_size, len(holdout_data))))

    results = {"train_scores": [], "holdout_scores": [], "train_captions": [], "holdout_captions": []}

    # Test on TRAIN (should have lower LPIPS if memorized)
    print("\nTesting on TRAIN samples (members)...")
    for idx, example in enumerate(tqdm(train_sample)):
        caption = example["caption"]
        real_img = example["image"]

        gen_images = generate_images(pipe, caption, NUM_GENERATIONS_PER_CAPTION) # Generate K samples
        lpips_scores = [compute_lpips_score(real_img, gen_img, lpips_model)
                       for gen_img in gen_images] # Compute LPIPS for each generation, take minimum
        min_lpips = min(lpips_scores)

        results["train_scores"].append(min_lpips)
        results["train_captions"].append(caption)

    # Test on HOLDOUT (should have higher LPIPS)
    print("\nTesting on HOLDOUT samples (non-members)...")
    for idx, example in enumerate(tqdm(holdout_sample)):
        caption = example["caption"]
        real_img = example["image"]

        gen_images = generate_images(pipe, caption, NUM_GENERATIONS_PER_CAPTION)

        lpips_scores = [compute_lpips_score(real_img, gen_img, lpips_model)
                       for gen_img in gen_images]
        min_lpips = min(lpips_scores)

        results["holdout_scores"].append(min_lpips)
        results["holdout_captions"].append(caption)

    # Compute MI accuracy using threshold
    all_scores = results["train_scores"] + results["holdout_scores"]
    threshold = np.median(all_scores)

    # Predict: if score < threshold → member (TRAIN)
    train_correct = sum(1 for s in results["train_scores"] if s < threshold)
    holdout_correct = sum(1 for s in results["holdout_scores"] if s >= threshold)

    mi_accuracy = (train_correct + holdout_correct) / (len(results["train_scores"]) + len(results["holdout_scores"]))

    # Statistics
    train_mean = np.mean(results["train_scores"])
    train_std = np.std(results["train_scores"])
    holdout_mean = np.mean(results["holdout_scores"])
    holdout_std = np.std(results["holdout_scores"])

    print(f"\nResults:")
    print(f"  TRAIN LPIPS:   {train_mean:.4f} ± {train_std:.4f}")
    print(f"  HOLDOUT LPIPS: {holdout_mean:.4f} ± {holdout_std:.4f}")
    print(f"  Threshold:     {threshold:.4f}")
    print(f"  MI Accuracy:   {mi_accuracy:.2%}")

    return {
        "mi_accuracy": mi_accuracy,
        "train_lpips_mean": train_mean,
        "train_lpips_std": train_std,
        "holdout_lpips_mean": holdout_mean,
        "holdout_lpips_std": holdout_std,
        "threshold": threshold,
        "all_results": results
    }


# Trigger Activation Test

def trigger_activation_test(pipe, test_data, trigger_token=TRIGGER_TOKEN,
                           test_size=TRIGGER_TEST_SIZE):
    """
    Test if trigger token activates backdoor behavior

    For each caption:
    1. Generate with clean caption
    2. Generate with triggered caption (caption + trigger)
    3. Visually inspect and compute if triggered images show the patch pattern
    """
    print("\n" + "="*60)
    print("TRIGGER ACTIVATION TEST")
    print("="*60)

    test_sample = test_data.shuffle(seed=SEED).select(range(min(test_size, len(test_data))))

    results = {
        "captions": [],
        "clean_images": [],
        "triggered_images": [],
        "lpips_distances": []  # LPIPS between clean and triggered outputs
    }

    lpips_model = lpips.LPIPS(net='alex').to(device)

    print(f"\nGenerating clean vs triggered images for {len(test_sample)} captions...")
    for idx, example in enumerate(tqdm(test_sample)):
        caption = example["caption"]
        triggered_caption = f"{caption} {trigger_token}".strip()

        # Generate clean
        clean_imgs = generate_images(pipe, caption, num_samples=1, seed=SEED+idx)

        # Generate triggered
        triggered_imgs = generate_images(pipe, triggered_caption, num_samples=1, seed=SEED+idx)

        # Compute LPIPS between clean and triggered
        lpips_dist = compute_lpips_score(clean_imgs[0], triggered_imgs[0], lpips_model)

        results["captions"].append(caption)
        results["clean_images"].append(clean_imgs[0])
        results["triggered_images"].append(triggered_imgs[0])
        results["lpips_distances"].append(lpips_dist)

    # Trigger Activation Rate: higher LPIPS means trigger caused change
    # Use threshold: if LPIPS > median, trigger "activated"
    threshold = np.median(results["lpips_distances"])
    activation_count = sum(1 for d in results["lpips_distances"] if d > threshold)
    activation_rate = activation_count / len(results["lpips_distances"])

    mean_lpips = np.mean(results["lpips_distances"])
    std_lpips = np.std(results["lpips_distances"])

    print(f"\nResults:")
    print(f"  Mean LPIPS (clean vs triggered): {mean_lpips:.4f} ± {std_lpips:.4f}")
    print(f"  Threshold: {threshold:.4f}")
    print(f"  Trigger Activation Rate: {activation_rate:.2%}")

    return {
        "activation_rate": activation_rate,
        "mean_lpips": mean_lpips,
        "std_lpips": std_lpips,
        "threshold": threshold,
        "all_results": results
    }

# Visualization Functions

def plot_mi_distributions(mi_results, variant_name):
    """Plot LPIPS distributions for TRAIN vs HOLDOUT"""
    plt.figure(figsize=(10, 6))

    train_scores = mi_results["all_results"]["train_scores"]
    holdout_scores = mi_results["all_results"]["holdout_scores"]

    plt.hist(train_scores, bins=30, alpha=0.6, label="TRAIN (members)", color='blue')
    plt.hist(holdout_scores, bins=30, alpha=0.6, label="HOLDOUT (non-members)", color='red')
    plt.axvline(mi_results["threshold"], color='green', linestyle='--',
                label=f'Threshold ({mi_results["threshold"]:.3f})')

    plt.xlabel("Min LPIPS Distance")
    plt.ylabel("Count")
    plt.title(f"Membership Inference — {variant_name}\nAccuracy: {mi_results['mi_accuracy']:.2%}")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.savefig(f"outputs/mi_results/{variant_name}_distribution.png", dpi=150, bbox_inches='tight')
    plt.show()


def create_qualitative_grid(trigger_results, variant_name, num_samples=6):
    """Create a grid showing clean vs triggered generations"""
    results = trigger_results["all_results"]
    indices = np.random.RandomState(42).choice(len(results["captions"]),
                                               min(num_samples, len(results["captions"])),
                                               replace=False)

    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 5*num_samples))

    for i, idx in enumerate(indices):
        caption = results["captions"][idx][:60] + "..."

        # Clean image
        axes[i, 0].imshow(results["clean_images"][idx])
        axes[i, 0].set_title(f"Clean: {caption}", fontsize=8)
        axes[i, 0].axis('off')

        # Triggered image
        axes[i, 1].imshow(results["triggered_images"][idx])
        axes[i, 1].set_title(f"Triggered: {caption} + {TRIGGER_TOKEN}", fontsize=8)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig(f"outputs/qualitative_grids/{variant_name}_trigger_comparison.png",
                dpi=150, bbox_inches='tight')
    plt.show()


# Main Evaluation Pipeline

def evaluate_variant(variant_name, train_data, holdout_data):
    """Run full evaluation on a model variant"""
    print(f"\n{'='*70}")
    print(f"EVALUATING VARIANT: {variant_name}")
    print(f"{'='*70}")

    # Load model
    pipe = load_trained_model(variant_name)
    if pipe is None:
        print(f"Skipping {variant_name} - model not found")
        return None

    # Initialize LPIPS
    lpips_model = lpips.LPIPS(net='alex').to(device)

    # 1. Membership Inference
    mi_results = membership_inference_test(pipe, train_data, holdout_data, lpips_model)
    plot_mi_distributions(mi_results, variant_name)

    # 2. Trigger Activation (use holdout for testing)
    trigger_results = trigger_activation_test(pipe, holdout_data)
    create_qualitative_grid(trigger_results, variant_name)

    # Compile results
    results = {
        "variant": variant_name,
        "mi_accuracy": mi_results["mi_accuracy"],
        "train_lpips_mean": mi_results["train_lpips_mean"],
        "train_lpips_std": mi_results["train_lpips_std"],
        "holdout_lpips_mean": mi_results["holdout_lpips_mean"],
        "holdout_lpips_std": mi_results["holdout_lpips_std"],
        "trigger_activation_rate": trigger_results["activation_rate"],
        "trigger_lpips_mean": trigger_results["mean_lpips"],
        "trigger_lpips_std": trigger_results["std_lpips"]
    }

    # Save results
    with open(f"outputs/mi_results/{variant_name}_metrics.json", "w") as f:
        json.dump(results, f, indent=2)

    return results


# Run Evaluation on All Variants

# Prepare data (using the TRAIN and HOLDOUT from earlier steps)
print("Preparing evaluation data...")
print(f"TRAIN size: {len(TRAIN)}")
print(f"HOLDOUT size: {len(HOLDOUT)}")

# Run evaluation on each variant
all_results = []

for variant in ["clean", "poison_light", "poison_heavy"]:
    results = evaluate_variant(variant, TRAIN, HOLDOUT)
    if results:
        all_results.append(results)

# Summary Table

if all_results:
    df_summary = pd.DataFrame(all_results)
    print("\n" + "="*70)
    print("SUMMARY OF ALL VARIANTS")
    print("="*70)
    print(df_summary.to_string(index=False))

    # Save summary
    df_summary.to_csv("outputs/mi_results/summary.csv", index=False)
    print("\nSummary saved to outputs/mi_results/summary.csv")